In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
import tempfile # Added import for tempfile

# Data preparation (using synthetic data for simulation)
def create_synthetic_dataset():
    """Create a synthetic dataset for recyclable items"""
    # Categories: plastic, paper, glass, metal
    categories = ['plastic', 'paper', 'glass', 'metal']

    # Generate synthetic images (in real scenario, use real dataset)
    num_samples = 1000
    img_height, img_width = 128, 128

    # Create random images with different patterns for each category
    X_train = []
    y_train = []

    for i in range(num_samples):
        category = i % len(categories)
        # Create different visual patterns for each category
        if category == 0:  # plastic - bottle-like shapes
            img = np.random.rand(img_height, img_width, 3) * 0.3
            # Add bottle-like vertical structure
            img[:, 50:80, :] += 0.4
        elif category == 1:  # paper - flat texture
            img = np.random.rand(img_height, img_width, 3) * 0.5
        elif category == 2:  # glass - transparent effect
            img = np.random.rand(img_height, img_width, 3) * 0.2
            img[30:100, 30:100, :] += 0.3
        else:  # metal - shiny effect
            img = np.random.rand(img_height, img_width, 3) * 0.4
            img[20:40, 20:120, :] += 0.3  # metallic strip

        X_train.append(img)
        y_train.append(category)

    return np.array(X_train), np.array(y_train), categories

# Create lightweight CNN model
def create_lightweight_model(input_shape=(128, 128, 3), num_classes=4):
    model = keras.Sequential([
        layers.Conv2D(16, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Train the model
print("Creating synthetic dataset...")
X_train, y_train, categories = create_synthetic_dataset()

print("Splitting data...")
split_idx = int(0.8 * len(X_train))
X_val, y_val = X_train[split_idx:], y_train[split_idx:]
X_train, y_train = X_train[:split_idx], y_train[:split_idx]

print("Creating model...")
model = create_lightweight_model()

print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

# Evaluate model
print("\nEvaluating model...")
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Convert to TensorFlow Lite
print("\nConverting to TensorFlow Lite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TensorFlow Lite model
with open('recyclable_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

print("TensorFlow Lite model saved as 'recyclable_classifier.tflite'")

# Test TensorFlow Lite model
print("\nTesting TensorFlow Lite model...")
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test on a few samples
correct_predictions = 0
total_samples = min(50, len(X_val))

for i in range(total_samples):
    test_image = X_val[i].astype(np.float32)
    true_label = y_val[i]

    # Set input tensor
    interpreter.set_tensor(input_details[0]['index'], [test_image])

    # Run inference
    interpreter.invoke()

    # Get prediction
    prediction = interpreter.get_tensor(output_details[0]['index'])
    predicted_label = np.argmax(prediction[0])

    if predicted_label == true_label:
        correct_predictions += 1

tflite_accuracy = correct_predictions / total_samples
print(f"TensorFlow Lite Test Accuracy: {tflite_accuracy:.4f}")

# Model size comparison
# Fix: Save weights to a temporary file to get its size
if hasattr(model, 'save_weights'):
    with tempfile.NamedTemporaryFile(suffix='.weights.h5', delete=False) as temp_weights_file:
        temp_path = temp_weights_file.name
    model.save_weights(temp_path)
    original_size = os.path.getsize(temp_path)
    os.remove(temp_path)
else:
    original_size = 0

tflite_size = len(tflite_model)

print(f"\nModel Size Comparison:")
print(f"Original Model: {original_size} bytes (estimated)")
print(f"TensorFlow Lite: {tflite_size} bytes")
print(f"Size Reduction: {(1 - tflite_size/original_size)*100:.1f}%" if original_size > 0 else "N/A")

Creating synthetic dataset...
Splitting data...
Creating model...
Training model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 13s 456ms/step - accuracy: 0.4950 - loss: 1.0812 - val_accuracy: 1.0000 - val_loss: 0.0158
Epoch 2/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 450ms/step - accuracy: 0.9987 - loss: 0.0206 - val_accuracy: 1.0000 - val_loss: 1.6719e-06
Epoch 3/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 443ms/step - accuracy: 0.9972 - loss: 0.0106 - val_accuracy: 1.0000 - val_loss: 8.2850e-08
Epoch 4/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 21s 445ms/step - accuracy: 0.9963 - loss: 0.0076 - val_accuracy: 1.0000 - val_loss: 2.5690e-07
Epoch 5/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 444ms/step - accuracy: 0.9982 - loss: 0.0082 - val_accuracy: 1.0000 - val_loss: 2.6870e-06
Epoch 6/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 442ms/step - accuracy: 0.9982 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 1.1086e-07
Epoch 7/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 421ms/step - accuracy: 0.9991 - loss: 0.0015 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 418ms/step - accuracy: 0.9989 -

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
